# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {}

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt  # lets us plot like MATLAB
import cv2
import os  # allows python script to interact with OS
import numpy as np
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,
)  # tools for real time data augmentation on image data
from tensorflow.keras.preprocessing import image

In [ ]:
img = image.load_img("data/train/triangles/drawing(1).PNG")

In [ ]:
plt.imshow(img)

In [ ]:
cv2.imread("data/train/triangles/drawing(1).PNG").shape  # 'shape' gives the dimensions of the image

In [ ]:
train = ImageDataGenerator(
    rescale=1 / 255
)  # this transforms the pixel RGB value from a range of 0-255 to a range of 0-1.000
validation = ImageDataGenerator(rescale=1 / 255)

In [ ]:
train_dataset = train.flow_from_directory(
    "data/train/", target_size=(28, 28), batch_size=50, class_mode="categorical"
)  # generates a batch of augmented data

In [ ]:
validation_dataset = validation.flow_from_directory(
    "data/validation/", target_size=(28, 28), batch_size=50, class_mode="categorical"
)  # generates a batch of augmented data

In [ ]:
train_dataset.class_indices  # checking indices of the classes

In [ ]:
from experiment_shape_classifier import build_model_cell_10

model = build_model_cell_10()

In [ ]:
model.compile(
    loss="categorical_crossentropy",  # chosing loss function and optimizer class
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

In [ ]:
model_fit = model.fit(
    train_dataset, steps_per_epoch=3, epochs=100, validation_data=validation_dataset
)

In [ ]:
dir_path = "data/test"

for i in os.listdir(dir_path):
    img = image.load_img(
        dir_path + "//" + i, target_size=(28, 28)
    )  # loads the test images and resizes them to target_size
    plt.imshow(img)
    plt.show()

    X = image.img_to_array(img)
    X = np.expand_dims(X, axis=0)
    images = np.vstack([X]) / 255.0
    val = model.predict(images)
    class_names = {value: key for key, value in train_dataset.class_indices.items()}
    print(class_names[int(np.argmax(val[0]))])